In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from config import TABLES

In [0]:
import importlib
import config

importlib.reload(config)

TABLES = config.TABLES

In [0]:
from pyspark.sql import functions as F

def log_pipeline_metric(spark, pipeline_name, df):

    metrics_df = spark.createDataFrame(
        [(pipeline_name, df.count())],
        ["pipeline", "row_count"]
    ).withColumn("processed_at", F.current_timestamp())

    metrics_df.write.mode("append").saveAsTable(
        "dbw_retails.monitoring.pipeline_metrics"
    )

In [0]:
# Enable schema auto merge
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

In [0]:
df_silver = spark.read.table(TABLES["silver"])

In [0]:
product_perf = (
    df_silver
    .groupBy("stock_code", "description")
    .agg(
        F.sum("quantity").alias("units_sold"),
        F.sum("revenue").alias("product_revenue"),
        F.countDistinct("invoice_no").alias("order_count")
    )
    .withColumn("updated_at", F.current_timestamp())
)

(
    product_perf.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TABLES["gold_product"])
)
log_pipeline_metric(spark, "gold_product", product_perf)

In [0]:
country_sales = (
    df_silver
    .groupBy("country", "year", "month")
    .agg(
        F.sum("revenue").alias("total_revenue"),
        F.sum("quantity").alias("items_sold"),
        F.countDistinct("invoice_no").alias("orders")
    )
    .withColumn("updated_at", F.current_timestamp())
)

(
    country_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLES["gold_country"])
)
log_pipeline_metric(spark, "gold_country", country_sales)

In [0]:
# ---------------------------------------------------
# 3. Customer Revenue
# ---------------------------------------------------

customer_revenue = (
    df_silver
    .filter(F.col("customer_id").isNotNull())   # remove null customers
    .groupBy("customer_id")
    .agg(
        F.countDistinct("invoice_no").alias("total_orders"),
        F.sum("quantity").alias("total_items"),
        F.sum("revenue").alias("total_revenue"),
        F.avg("revenue").alias("avg_order_value")
    )
    .withColumn("updated_at", F.current_timestamp())
)

(
    customer_revenue.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLES["gold_customer"])
)
log_pipeline_metric(spark, "gold_customer", customer_revenue)

In [0]:
print("Gold layer incremental pipeline completed successfully.")

Gold layer incremental pipeline completed successfully.
